## HMM Distillation Tutorial

To distill a hmm model, we need to 
* (1) prepare the input data for sampling
* (2) sample the output
* (3) Initialize the checkpoint-0 with lvd
* (4) train the hmm model.

### Basic Config

In [ ]:
import os

# Sampling Dataset Path: The path to save the dataset used for sampling
SAMPLING_DATASET_PATH = "dapo_prompts_boxed_shuffled.json"
# Model path or model name from Hugging Face
# BASE_MODEL_PATH = "billkunghappy/Qwen3-8B-Base-Dapo-V7-S60"
BASE_MODEL_PATH = "billkunghappy/Qwen3-1.7B-Base-Dapo-V1-S60"
# The name used to save/load the model and data
MODEL_NAME="DAPO-DAPO-Baseline-Qwen3-1.7B-V1"

DATASET="dapo"
OUTPUT_DIR = f'./{DATASET}/{MODEL_NAME}'

try:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Directory '{OUTPUT_DIR}' created successfully.")
except OSError as e:
    print(f"Error creating directory: {e}")

### Step 1: Preproc the sampling dataset

In [ ]:
# Here we use DAPO-Math-17k as an example
import json
import datasets
import random

data_path = "open-r1/DAPO-Math-17k-Processed"
dataset = datasets.load_dataset(data_path, "all",split="train")

sample_num = 2000

all_prompts = []
for d in dataset:
    question = d["prompt"]
    prompt = f"{question}\nPlease reason step by step, and put your final answer within \\boxed{{}}.\n"
    all_prompts.append(prompt)

random.shuffle(all_prompts)

with open(SAMPLING_DATASET_PATH, "w") as f:
    json.dump(all_prompts[:sample_num], f, indent=4)

### Step 2: Sample the output from the base model

In [ ]:
# Launch the vllm server to sample the output from the base model
GPUS = "2,3,6,7"
cmd = f"./launch_vllm_model.sh {GPUS} {BASE_MODEL_PATH}"

print(cmd)

In [ ]:
# After the server is launched, we can use the following code to sample the output from the base model
IS_VLM = False

cmd = f"./sample_data_vllm.sh \
    {BASE_MODEL_PATH} \
    {MODEL_NAME} \
    {SAMPLING_DATASET_PATH} \
    {OUTPUT_DIR} \
    {IS_VLM}"

print(cmd)

In [ ]:
# Get the lvd data
GPUS = "2,3,5,6"
LVD_PATH = f"{OUTPUT_DIR}/{MODEL_NAME}.lvd"

cmd = f"./run_get_lvd_embedding.sh \
    {LVD_PATH} \
    {BASE_MODEL_PATH} \
    {GPUS}"
print(cmd)

### Step 3: Initialize hmm with lvd

In [ ]:
import os
from transformers import AutoModelForCausalLM, AutoTokenizer


# specify the HMM size
HIDDEN_STATES = 4096

# get vocab_size and eos_token_id; might vary for different models #
__tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True)
__model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True)
VOCAB_SIZE = __model.config.vocab_size # will be different to __tokenizer.vocab_size
EOS_TOKEN_ID = __tokenizer.eos_token_id
####################################################################

HMM_MODEL_ID = f'hmm_{MODEL_NAME}_{DATASET}_{HIDDEN_STATES}'
HMM_MODEL_PATH = f'./workspace/models/{HMM_MODEL_ID}'

_ = os.system(f'mkdir -p {HMM_MODEL_PATH}')

In [ ]:
import os

GPUS = '0,1,2,3'
SEQUENCES_FILE = f'{OUTPUT_DIR}/{MODEL_NAME}.lvd'
EMEBEDDINGS_FILE = f'{OUTPUT_DIR}/{MODEL_NAME}.lvd.embeddings.safetensors'

# latent variable distillation
cmd = f'CUDA_VISIBLE_DEVICES={GPUS} python lvd_hmm.py \
    --sequences_file {SEQUENCES_FILE} --embeddings_file {EMEBEDDINGS_FILE} \
    --hidden_states {HIDDEN_STATES} --vocab_size {VOCAB_SIZE} --eos_token_id {EOS_TOKEN_ID} \
    --kmeans_iterations 100 --pseudocount 0.001 \
    --output_file {HMM_MODEL_PATH}/checkpoint-0'
print(cmd)

### Step 4: Train HMM via Expectation Maximization (EM)

In [ ]:
import os

os.system('mkdir -p ./workspace/logs')
LOG_FILE=f'./workspace/logs/{HMM_MODEL_ID}_log.txt'

CUDA_CORES = '0,1,2,3'
TOTAL_CHUNKS = 100
BATCH_SIZE = 512
SAVE_PER_STEP = 10
DROPOUT = 0.01

# EM training schedule:
# 1. train for 100 EM steps, each step using 1 chunk of data
# 2. train for 50 EM steps, each step using 2 chunks of data
# 3. train for 40 EM steps, each step using 5 chunks of data
# 4. train for 40 EM steps, each step using 10 chunks of data
# 5. train for 40 EM steps, each step using 20 chunks of data
# 6. train for 10 EM steps, each step using 40 chunks of data
EM_SCHEDULE = "\"100,1;50,2;40,5;40,10;40,20;10,40\""

cmd = f'CUDA_VISIBLE_DEVICES={CUDA_CORES} torchrun --standalone --nproc_per_node=gpu train_hmm.py \
    --model_path {HMM_MODEL_PATH} --checkpoint 0 --save_per_step {SAVE_PER_STEP} \
    --data_path {OUTPUT_DIR} --dataset {MODEL_NAME} --total_chunks {TOTAL_CHUNKS} --batch_size {BATCH_SIZE} \
    --em_schedule {EM_SCHEDULE} --dropout {DROPOUT} --log_file {LOG_FILE}'.strip()
print(cmd)